# Project 03 · Quantitative Trading Strategies Lab

**Course:** Algorithmic Trading with Python · **Companion to:** *Lecture 03 – Trading Strategies & Paradigms*

---

## Welcome to the Practical Lab!
In Lecture 03, you explored the theoretical foundations of trading paradigms: **Trend-Following**, **Momentum**, and **Mean-Reversion**. 

In this lab, you will **implement your own quantitative trading strategies from scratch** using Python and `pandas`. Rather than relying on rigid canned scripts, you will learn the exact quantitative engineering workflow used by systematic traders:

1. **Rule Definition:** Translate plain-English market hypotheses into vectorized mathematical rules.
2. **Indicator Engineering:** Build robust technical indicators (Donchian Channels, Exponential Moving Averages, Bollinger Z-Scores, and RSI) from first principles.
3. **Signal & Position Architecture:** Avoid the **#1 rookie mistake in algorithmic trading—lookahead bias**—by implementing strict trade execution lag.
4. **Frictional Backtesting:** Model real-world slippage and transaction costs so your strategy is tested under realistic market friction.
5. **Risk-Adjusted Performance:** Measure **CAGR**, **Annualized Volatility**, **Sharpe Ratio**, and **Maximum Drawdown (MDD)** to objectively compare strategies.
6. **Capstone Challenge:** Architect and backtest your own proprietary hybrid strategy!

---

### How This Lab is Structured
- **80/20 Notes:** High-yield summaries covering the 20% of core concepts that drive 80% of practical algorithmic trading tasks.
- **Progressive Tasks:** Step-by-step `# TODO` blocks with guidance, hints, and expected outputs.
- **Self-Check Cells:** Built-in `assert` checks to verify your code before you proceed.
- **Independent Design:** You are encouraged to tweak parameters, test alternative rules, and develop your own trading intuition.

## Part 0 · Environment Setup & The Multi-Regime Market Dataset

### 80/20 Short Notes: Market Regimes & Offline Unit Testing
- Real financial markets are non-stationary: they cycle through **trending regimes**, **choppy sideways regimes**, and **high-volatility stress regimes**.
- When designing trading algorithms, professional quants first test logic on controlled, reproducible datasets with known statistical properties before running historical backtests.
- In this lab, we use a pre-generated 500-day dataset (`data/simulated_market.csv`) engineered with three distinct market regimes:
  1. **Days 1–200:** Strong upward trend (ideal for trend-followers).
  2. **Days 201–350:** Sideways choppy oscillation (ideal for mean-reversion, treacherous for trend-followers).
  3. **Days 351–500:** Volatile trending market (tests risk management and drawdown resistance).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook plotting setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_PATH = "data/simulated_market.csv"

# Load the dataset (or generate if running standalone)
if os.path.exists(DATA_PATH):
    df_market = pd.read_csv(DATA_PATH, index_col="Date", parse_dates=True)
else:
    # Fallback generator if CSV is missing
    np.random.seed(42)
    dates = pd.date_range("2023-01-02", periods=500, freq="B")
    r1 = np.random.normal(0.0012, 0.011, 200)
    r2 = np.random.normal(0.0000, 0.014, 150)
    r3 = np.random.normal(0.0007, 0.020, 150)
    shocks = np.concatenate([r1, r2, r3])
    close = 100.0 * np.cumprod(1 + shocks)
    df_market = pd.DataFrame(index=dates)
    df_market.index.name = "Date"
    df_market["Close"] = np.round(close, 2)
    df_market["High"] = np.round(df_market["Close"] * (1 + np.abs(np.random.normal(0, 0.006, 500))), 2)
    df_market["Low"] = np.round(df_market["Close"] * (1 - np.abs(np.random.normal(0, 0.006, 500))), 2)
    df_market["Open"] = np.round(df_market["Close"].shift(1).fillna(df_market["Close"].iloc[0]), 2)
    df_market["High"] = np.maximum(df_market["High"], np.maximum(df_market["Open"], df_market["Close"]))
    df_market["Low"] = np.minimum(df_market["Low"], np.minimum(df_market["Open"], df_market["Close"]))
    df_market["Volume"] = np.random.randint(500000, 2500000, size=500)

print(f"Market dataset loaded successfully: {len(df_market)} trading days.")
print(df_market.head())

In [ ]:
# Visualizing the 3 Market Regimes
plt.figure(figsize=(13, 5))
plt.plot(df_market.index, df_market['Close'], label='Market Close Price', color='#1f77b4', lw=1.8)
plt.axvspan(df_market.index[0], df_market.index[199], color='green', alpha=0.08, label='Regime 1: Upward Trend')
plt.axvspan(df_market.index[200], df_market.index[349], color='orange', alpha=0.08, label='Regime 2: Sideways Chop')
plt.axvspan(df_market.index[350], df_market.index[-1], color='red', alpha=0.08, label='Regime 3: Volatile Trend')
plt.title('Asset Price with Distinct Market Regimes', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

## Part 1 · Signal Architecture & The "Cardinal Sin" (Lookahead Bias)

### 80/20 Short Notes: Signal vs. Position
Every systematic trading algorithm follows a strict 5-stage pipeline:
$$\text{Market Data} \longrightarrow \text{Indicator} \longrightarrow \text{Raw Signal} \longrightarrow \text{Position} \longrightarrow \text{Returns}$$

- **Signal ($S_t$):** The trading decision formed after evaluating day $t$'s market data at the close (e.g., $+1$ for Long, $0$ for Flat/Cash, $-1$ for Short).
- **Position ($P_t$):** The actual portfolio holding during day $t$.
- **The Golden Execution Rule:**
  $$\mathbf{Position_t = Signal_{t-1}}$$
  In pandas: `df['Position'] = df['Signal'].shift(1)`

### Why is `.shift(1)` Non-Negotiable?
If you evaluate today's closing price at 4:00 PM, you cannot travel back in time to buy at 9:30 AM or at today's 4:00 PM close. Your trade executes on the **next bar** ($t+1$). If you multiply today's signal by today's return without shifting, you introduce **lookahead bias** (peeking into the future), producing astronomical yet completely fictional backtest returns.

### Task 1.1: Naive Price vs. 50-day Moving Average Signal
1. Calculate a 50-day Simple Moving Average (`SMA_50`) on `Close`.
2. Generate a raw trading signal:
   - `Signal = 1` when `Close > SMA_50`
   - `Signal = 0` when `Close <= SMA_50`
3. Generate the actionable holding position using `.shift(1)`.

In [ ]:
df_p1 = df_market.copy()

# 1.1: Calculate 50-day SMA
df_p1['SMA_50'] = df_p1['Close'].rolling(window=50).mean()

# 1.2: Generate Raw Signal (1 if Close > SMA_50, else 0)
df_p1['Signal'] = np.where(df_p1['Close'] > df_p1['SMA_50'], 1, 0)

# 1.3: Apply 1-day execution lag to avoid lookahead bias
df_p1['Position'] = df_p1['Signal'].shift(1).fillna(0)

print(df_p1[['Close', 'SMA_50', 'Signal', 'Position']].tail())

In [ ]:
# Self-Check for Part 1
assert 'SMA_50' in df_p1.columns, "Missing 'SMA_50' column"
assert 'Signal' in df_p1.columns, "Missing 'Signal' column"
assert 'Position' in df_p1.columns, "Missing 'Position' column"
assert df_p1['Position'].iloc[0] == 0, "Position on first row should be 0 (no signal yet)"
# Verify execution lag: Position at index i must equal Signal at index i-1
assert (df_p1['Position'].iloc[51:] == df_p1['Signal'].iloc[50:-1].values).all(), "Position is not shifted by 1 relative to Signal!"
print(" Part 1 assertions passed! Position lag correctly implemented.")

## Part 2 · Trend Following Paradigm: Donchian Channel Breakout

### 80/20 Short Notes: Breakout Trading
In Lecture 03, trend-based strategies include moving averages and **breakout strategies**.
- Breakout trading originated with Richard Donchian and the legendary Turtle Traders.
- Rather than guessing bottoms, a breakout strategy buys when the price crosses above the highest high of the past $N$ days, signaling strong institutional buying momentum.
- **Rules:**
  1. **Upper Channel ($UC_t$):** Highest price of the previous $N$ days.
  2. **Lower Channel ($LC_t$):** Lowest price of the previous $N$ days.
  3. **Long Entry:** If $Close_t > UC_t$, set Signal = $+1$.
  4. **Exit to Cash:** If $Close_t < LC_t$, set Signal = $0$.
  5. **Holding Rule:** If price is between channels, maintain previous position (`ffill()`).

*Critical Implementation Detail:* To prevent lookahead bias when calculating channels, you must calculate the rolling max/min on `.shift(1)` so today's close is not compared against itself!

### Task 2: Implement the 20-Day Donchian Breakout
1. Calculate `Donchian_High`: The rolling 20-day maximum of `Close`, shifted by 1 day.
2. Calculate `Donchian_Low`: The rolling 20-day minimum of `Close`, shifted by 1 day.
3. Formulate the stateful signal:
   - $+1$ when `Close > Donchian_High`
   - $0$ when `Close < Donchian_Low`
   - `np.nan` otherwise, then forward-fill (`.ffill()`) and fill initial NaNs with $0$.
4. Shift the signal by 1 bar to obtain `Position_Breakout`.

In [ ]:
df_p2 = df_market.copy()
WINDOW = 20

# 2.1: Donchian Channels (shifted by 1 to exclude current bar)
df_p2['Donchian_High'] = df_p2['Close'].shift(1).rolling(window=WINDOW).max()
df_p2['Donchian_Low'] = df_p2['Close'].shift(1).rolling(window=WINDOW).min()

# 2.2: Raw signal state
raw_sig = np.where(
    df_p2['Close'] > df_p2['Donchian_High'], 1,
    np.where(df_p2['Close'] < df_p2['Donchian_Low'], 0, np.nan)
)
df_p2['Signal_Breakout'] = pd.Series(raw_sig, index=df_p2.index).ffill().fillna(0)

# 2.3: Actionable position with 1-bar execution lag
df_p2['Position_Breakout'] = df_p2['Signal_Breakout'].shift(1).fillna(0)

print(df_p2[['Close', 'Donchian_High', 'Donchian_Low', 'Signal_Breakout', 'Position_Breakout']].dropna().head())

In [ ]:
# Self-Check for Part 2
assert 'Donchian_High' in df_p2.columns and 'Donchian_Low' in df_p2.columns
assert set(df_p2['Signal_Breakout'].unique()).issubset({0, 1}), "Signals must only be 0 or 1"
assert (df_p2['Position_Breakout'].iloc[1:] == df_p2['Signal_Breakout'].iloc[:-1].values).all(), "Position must lag Signal by 1 bar"
print(f" Part 2 assertions passed! Total breakout position changes: {(df_p2['Position_Breakout'].diff().abs() > 0).sum()}")

## Part 3 · Dual Moving Average Paradigm: EMA with Volatility Buffer

### 80/20 Short Notes: The Moving Average Whipsaw Problem
- In Lecture 03, moving average crossover was introduced as the foundational algorithmic strategy.
- **SMA vs. EMA:** Simple Moving Average weights all $N$ days equally. Exponential Moving Average (**EMA**) applies exponentially declining weights, reacting faster to recent trend shifts:
  $$\text{EMA}_t = \alpha \cdot P_t + (1 - \alpha) \cdot \text{EMA}_{t-1}$$
  In pandas: `df['Close'].ewm(span=N, adjust=False).mean()`
- **The Whipsaw Problem:** In sideways or choppy markets, fast and slow moving averages repeatedly criss-cross every few days. This results in heavy losses from false entries and transaction fees.
- **The Solution (Buffer Band):** Only trigger a Long signal when the Fast EMA leads the Slow EMA by a safety margin (e.g., at least $+0.5\%$ spread).

### Task 3: Implement Buffered Dual-EMA Crossover
1. Calculate `EMA_Fast` (12-day span) and `EMA_Slow` (26-day span).
2. Calculate the normalized spread:
   $$\text{Spread}_t = \frac{\text{EMA\_Fast}_t - \text{EMA\_Slow}_t}{\text{EMA\_Slow}_t}$$
3. Generate signals using a buffer threshold of $\pm 0.005$ ($0.5\%$):
   - $+1$ if $\text{Spread} > +0.005$ (Confirmed Uptrend)
   - $0$ if $\text{Spread} < -0.005$ (Confirmed Downtrend/Exit)
   - `np.nan` otherwise (maintain existing position via `.ffill()`)
4. Create `Position_EMA` with `.shift(1)`.

In [ ]:
df_p3 = df_market.copy()
FAST_SPAN = 12
SLOW_SPAN = 26
BUFFER = 0.005  # 0.5% threshold

# 3.1: Fast and Slow EMAs
df_p3['EMA_Fast'] = df_p3['Close'].ewm(span=FAST_SPAN, adjust=False).mean()
df_p3['EMA_Slow'] = df_p3['Close'].ewm(span=SLOW_SPAN, adjust=False).mean()

# 3.2: Percentage spread
df_p3['EMA_Spread'] = (df_p3['EMA_Fast'] - df_p3['EMA_Slow']) / df_p3['EMA_Slow']

# 3.3: Buffered signal
raw_ema_sig = np.where(
    df_p3['EMA_Spread'] > BUFFER, 1,
    np.where(df_p3['EMA_Spread'] < -BUFFER, 0, np.nan)
)
df_p3['Signal_EMA'] = pd.Series(raw_ema_sig, index=df_p3.index).ffill().fillna(0)

# 3.4: Actionable position
df_p3['Position_EMA'] = df_p3['Signal_EMA'].shift(1).fillna(0)

print(df_p3[['Close', 'EMA_Fast', 'EMA_Slow', 'EMA_Spread', 'Signal_EMA', 'Position_EMA']].tail())

In [ ]:
# Self-Check for Part 3
assert 'EMA_Fast' in df_p3.columns and 'EMA_Slow' in df_p3.columns
assert (df_p3['Position_EMA'].iloc[1:] == df_p3['Signal_EMA'].iloc[:-1].values).all()
print(f" Part 3 assertions passed! Filtered EMA position changes: {(df_p3['Position_EMA'].diff().abs() > 0).sum()}")

## Part 4 · Mean Reversion Paradigm: Rolling Z-Score with Asymmetric Exit

### 80/20 Short Notes: Statistical Mean Reversion
- Lecture 03 introduced mean reversion: asset prices deviate from their equilibrium mean and tend to revert back over time.
- **The Z-Score Indicator:** Standardizes price deviation in terms of rolling standard deviations:
  $$Z_t = \frac{\text{Close}_t - \mu_t}{\sigma_t}$$
  where $\mu_t$ is the 20-day rolling mean and $\sigma_t$ is the 20-day rolling standard deviation.
- **Asymmetric Rules for Long-Only Strategy:**
  1. **Entry (Oversold):** When $Z_t < -1.5$, price is statistically depressed. Go Long ($+1$).
  2. **Exit (Mean Reached):** When $Z_t \ge 0.0$, price has returned to its average. Exit to Cash ($0$).
  3. **Holding:** Maintain state until an entry or exit condition is satisfied.

### Task 4: Implement Z-Score Mean Reversion
1. Compute 20-day rolling mean `Roll_Mean` and standard deviation `Roll_Std`.
2. Compute `Z_Score` = `(Close - Roll_Mean) / Roll_Std`.
3. Generate the signal:
   - $+1$ when `Z_Score < -1.5`
   - $0$ when `Z_Score >= 0.0`
   - `np.nan` otherwise, then forward fill.
4. Shift by 1 to create `Position_Z`.

In [ ]:
df_p4 = df_market.copy()
Z_WINDOW = 20

# 4.1: Rolling mean and rolling std
df_p4['Roll_Mean'] = df_p4['Close'].rolling(window=Z_WINDOW).mean()
df_p4['Roll_Std'] = df_p4['Close'].rolling(window=Z_WINDOW).std()

# 4.2: Z-Score
df_p4['Z_Score'] = (df_p4['Close'] - df_p4['Roll_Mean']) / df_p4['Roll_Std']

# 4.3: Asymmetric signal (+1 when oversold, 0 when reverted to mean)
raw_z_sig = np.where(
    df_p4['Z_Score'] < -1.5, 1,
    np.where(df_p4['Z_Score'] >= 0.0, 0, np.nan)
)
df_p4['Signal_Z'] = pd.Series(raw_z_sig, index=df_p4.index).ffill().fillna(0)

# 4.4: Actionable position
df_p4['Position_Z'] = df_p4['Signal_Z'].shift(1).fillna(0)

print(df_p4[['Close', 'Roll_Mean', 'Roll_Std', 'Z_Score', 'Signal_Z', 'Position_Z']].dropna().head())

In [ ]:
# Self-Check for Part 4
assert 'Z_Score' in df_p4.columns
assert not df_p4['Z_Score'].dropna().empty
assert (df_p4['Position_Z'].iloc[1:] == df_p4['Signal_Z'].iloc[:-1].values).all()
print(f" Part 4 assertions passed! Z-Score mean reversion trades: {(df_p4['Position_Z'].diff().abs() > 0).sum()}")

## Part 5 · Momentum Paradigm: Relative Strength Index (RSI) from Scratch

### 80/20 Short Notes: Demystifying RSI
In Lecture 03, the Relative Strength Index (RSI) was introduced. Many beginners treat it as a black box inside third-party packages, but building it directly in `pandas` reveals its intuitive mechanics:
1. **Price Difference:** $\Delta_t = \text{Close}_t - \text{Close}_{t-1}$
2. **Gain & Loss Series:**
   $$\text{Gain}_t = \max(\Delta_t, 0), \quad \text{Loss}_t = \max(-\Delta_t, 0)$$
3. **Smoothed Exponential Averages ($N=14$):**
   $$\overline{\text{Gain}} = \text{EMA}(\text{Gain}, \alpha=1/14), \quad \overline{\text{Loss}} = \text{EMA}(\text{Loss}, \alpha=1/14)$$
4. **Relative Strength ($RS$) & RSI (bounded between 0 and 100):**
   $$RS = \frac{\overline{\text{Gain}}}{\overline{\text{Loss}}}, \quad RSI = 100 - \frac{100}{1 + RS}$$

- **Trading Rule:**
  - Buy when $RSI < 30$ (Oversold extreme).
  - Take profit / Exit when $RSI > 65$ (Overbought / momentum exhaustion).

### Task 5: Calculate RSI & Generate Strategy Positions
1. Calculate daily price change $\Delta$ using `.diff()`.
2. Split $\Delta$ into positive `Gain` and absolute negative `Loss` (hint: use `.clip(lower=0)` and `.clip(upper=0)`).
3. Smooth both using `.ewm(alpha=1/14, adjust=False).mean()`.
4. Calculate $RS$ and the final $RSI$ series.
5. Generate the signal ($+1$ if $RSI < 30$, $0$ if $RSI > 65$, `np.nan` otherwise with `.ffill()`).
6. Shift by 1 to create `Position_RSI`.

In [ ]:
df_p5 = df_market.copy()
RSI_PERIOD = 14

# 5.1: Price difference
delta = df_p5['Close'].diff()

# 5.2: Separate gains and losses
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

# 5.3: Exponential smoothing with alpha = 1 / 14 (Wilder's smoothing)
avg_gain = gain.ewm(alpha=1/RSI_PERIOD, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/RSI_PERIOD, adjust=False).mean()

# 5.4: Relative Strength and RSI formula
rs = avg_gain / avg_loss.replace(0, np.nan)
df_p5['RSI'] = 100 - (100 / (1 + rs))

# 5.5: Strategy signal
raw_rsi_sig = np.where(
    df_p5['RSI'] < 30, 1,
    np.where(df_p5['RSI'] > 65, 0, np.nan)
)
df_p5['Signal_RSI'] = pd.Series(raw_rsi_sig, index=df_p5.index).ffill().fillna(0)

# 5.6: Actionable position
df_p5['Position_RSI'] = df_p5['Signal_RSI'].shift(1).fillna(0)

print(df_p5[['Close', 'RSI', 'Signal_RSI', 'Position_RSI']].dropna().tail())

In [ ]:
# Self-Check for Part 5
assert 'RSI' in df_p5.columns
valid_rsi = df_p5['RSI'].dropna()
assert (valid_rsi >= 0).all() and (valid_rsi <= 100).all(), "RSI values must be strictly bounded between 0 and 100"
assert (df_p5['Position_RSI'].iloc[1:] == df_p5['Signal_RSI'].iloc[:-1].values).all()
print(f" Part 5 assertions passed! RSI range: [{valid_rsi.min():.1f}, {valid_rsi.max():.1f}]. Position changes: {(df_p5['Position_RSI'].diff().abs() > 0).sum()}")

## Part 6 · Building a Reusable Backtesting Engine with Friction

### 80/20 Short Notes: Vectorized Backtesting & Frictions
In Lecture 03, naive backtesting was introduced by testing SMA periods. To build a robust framework, your backtesting engine must incorporate:
1. **Asset Return:** $R_{asset, t} = \frac{Close_t}{Close_{t-1}} - 1$
2. **Gross Strategy Return:** $R_{gross, t} = Position_t \times R_{asset, t}$
3. **Turnover & Transaction Costs:**
   Every time your position changes (e.g., from $0$ to $1$, or $1$ to $0$), you execute a trade. 
   $$\text{Trade Volume}_t = |Position_t - Position_{t-1}|$$
   $$\text{Fee}_t = \text{Trade Volume}_t \times \text{Fee Rate}$$
   A typical institutional commission + bid-ask slippage estimate is **10 basis points (0.10% or 0.0010)** per trade.
4. **Net Strategy Return & Wealth Curve:**
   $$R_{net, t} = R_{gross, t} - \text{Fee}_t$$
   $$W_t = \prod_{i=1}^t (1 + R_{net, i}) \quad \text{(Starting from } W_0 = 1.0\text{)}$$

### Task 6: Implement `backtest_strategy()`
Write a reusable function `backtest_strategy(df, position_col, fee_bps=10)` that:
1. Calculates asset percentage returns.
2. Calculates gross strategy return (`Position * Asset_Return`).
3. Calculates transaction fees based on position changes (`diff().abs()`).
4. Computes net return and the cumulative wealth series.
5. Returns a DataFrame with columns `['Net_Return', 'Wealth']`.

In [ ]:
def backtest_strategy(df, position_col, fee_bps=10):
    """
    Vectorized backtesting engine modeling position returns and transaction friction.
    
    Parameters:
    -----------
    df : pd.DataFrame with 'Close' column and position_col
    position_col : str, name of the column containing position (0, 1)
    fee_bps : int or float, transaction cost in basis points (10 bps = 0.0010)
    
    Returns:
    --------
    results : pd.DataFrame with 'Net_Return' and 'Wealth'
    """
    results = pd.DataFrame(index=df.index)
    fee_rate = fee_bps / 10000.0
    
    # 6.1: Asset returns
    asset_ret = df['Close'].pct_change().fillna(0)
    
    # 6.2: Gross strategy returns
    gross_ret = df[position_col] * asset_ret
    
    # 6.3: Turnover (trade count / volume)
    trade_volume = df[position_col].diff().abs().fillna(0)
    
    # 6.4: Fees
    fee = trade_volume * fee_rate
    
    # 6.5: Net return & Cumulative wealth
    results['Net_Return'] = gross_ret - fee
    results['Wealth'] = (1 + results['Net_Return']).cumprod()
    
    return results

# Test run on Breakout strategy
res_test = backtest_strategy(df_p2, 'Position_Breakout', fee_bps=10)
print(res_test.tail())

In [ ]:
# Self-Check for Part 6
assert 'Net_Return' in res_test.columns and 'Wealth' in res_test.columns
assert res_test['Wealth'].iloc[0] == 1.0 or abs(res_test['Wealth'].iloc[0] - 1.0) < 1e-4
print(f" Part 6 assertions passed! Final wealth for test breakout: {res_test['Wealth'].iloc[-1]:.3f}")

## Part 7 · Quantitative Risk & Performance Metrics

### 80/20 Short Notes: Evaluating Strategies Beyond Total Return
In finance, return without context is dangerous. A strategy that makes $40\%$ but suffers a $-50\%$ drawdown will cause panic or margin calls. Quants use four standard metrics:

1. **Compound Annual Growth Rate (CAGR):**
   $$\text{CAGR} = (W_T)^{\frac{252}{N}} - 1$$
   where $W_T$ is the final wealth and $N$ is total trading days (using 252 trading days per year).
2. **Annualized Volatility:**
   $$\sigma_{ann} = \text{std}(R_{net}) \times \sqrt{252}$$
3. **Sharpe Ratio (Risk-Adjusted Return):**
   $$\text{Sharpe} = \frac{\text{CAGR} - R_f}{\sigma_{ann}} \quad (R_f = 0 \text{ for simplicity})$$
4. **Maximum Drawdown (MDD):**
   The worst peak-to-trough decline experienced:
   $$\text{Peak}_t = \max_{s \le t} W_s, \quad \text{Drawdown}_t = \frac{W_t - \text{Peak}_t}{\text{Peak}_t}$$
   $$\text{MDD} = \min_t (\text{Drawdown}_t)$$

### Task 7: Implement `calculate_metrics()`
Write a function that accepts `wealth_series`, `daily_returns`, and a `name` string, returning a clean dictionary of key quantitative metrics:
- Total Return (%)
- CAGR (%)
- Annualized Volatility (%)
- Sharpe Ratio
- Maximum Drawdown (%)

In [ ]:
def calculate_metrics(wealth_series, daily_returns, name="Strategy"):
    """
    Computes annualized quantitative performance and risk metrics.
    """
    n_days = len(daily_returns)
    total_return = wealth_series.iloc[-1] - 1.0
    
    # 7.1: CAGR
    cagr = (wealth_series.iloc[-1]) ** (252 / n_days) - 1.0
    
    # 7.2: Annualized Volatility
    ann_vol = daily_returns.std() * np.sqrt(252)
    
    # 7.3: Sharpe Ratio (rf = 0)
    sharpe = cagr / ann_vol if ann_vol > 0 else 0.0
    
    # 7.4: Maximum Drawdown
    peak = wealth_series.cummax()
    drawdown = (wealth_series - peak) / peak
    max_dd = drawdown.min()
    
    return {
        "Strategy": name,
        "Total Return (%)": round(total_return * 100, 2),
        "CAGR (%)": round(cagr * 100, 2),
        "Ann. Volatility (%)": round(ann_vol * 100, 2),
        "Sharpe Ratio": round(sharpe, 2),
        "Max Drawdown (%)": round(max_dd * 100, 2)
    }

# Quick test
metrics_test = calculate_metrics(res_test['Wealth'], res_test['Net_Return'], "Test Breakout")
print(pd.Series(metrics_test))

In [ ]:
# Self-Check for Part 7
assert "Sharpe Ratio" in metrics_test and "Max Drawdown (%)" in metrics_test
assert metrics_test["Max Drawdown (%)"] <= 0.0, "Max drawdown should be non-positive"
print(" Part 7 assertions passed! Performance metrics engine verified.")

## Part 8 · The Showdown: Multi-Strategy Benchmark Comparison

Now we bring all our components together! We will backtest:
1. **Benchmark:** Buy-and-Hold (always 100% long)
2. **Strategy 1:** Donchian Breakout (Trend Following)
3. **Strategy 2:** Buffered Dual-EMA (Trend Following with Filter)
4. **Strategy 3:** Z-Score Mean Reversion (Statistical Contrarian)
5. **Strategy 4:** RSI Pullback (Momentum Exhaustion)

All strategies are subjected to the same **10 bps transaction fee** across all 500 trading days.

In [ ]:
# 1. Benchmark (Buy & Hold)
df_market['Position_BnH'] = 1.0
bnh_res = backtest_strategy(df_market, 'Position_BnH', fee_bps=10)

# 2. Donchian Breakout
breakout_res = backtest_strategy(df_p2, 'Position_Breakout', fee_bps=10)

# 3. Buffered Dual EMA
ema_res = backtest_strategy(df_p3, 'Position_EMA', fee_bps=10)

# 4. Z-Score Mean Reversion
z_res = backtest_strategy(df_p4, 'Position_Z', fee_bps=10)

# 5. RSI Pullback
rsi_res = backtest_strategy(df_p5, 'Position_RSI', fee_bps=10)

# Build Performance Comparison Table
strategies = [
    (bnh_res, "Buy & Hold (Benchmark)"),
    (breakout_res, "Donchian Breakout"),
    (ema_res, "Buffered Dual-EMA"),
    (z_res, "Z-Score Mean Reversion"),
    (rsi_res, "RSI Pullback")
]

summary_rows = [calculate_metrics(res['Wealth'], res['Net_Return'], name) for res, name in strategies]
df_summary = pd.DataFrame(summary_rows).set_index("Strategy")
display(df_summary)

In [ ]:
# Plotting the Comparative Equity Curves
plt.figure(figsize=(14, 7))
plt.plot(bnh_res['Wealth'], label='Buy & Hold', color='black', linestyle='--', alpha=0.7, lw=1.5)
plt.plot(breakout_res['Wealth'], label='Donchian Breakout', color='#1f77b4', lw=2)
plt.plot(ema_res['Wealth'], label='Buffered Dual-EMA', color='#2ca02c', lw=2)
plt.plot(z_res['Wealth'], label='Z-Score Mean Reversion', color='#d62728', lw=2)
plt.plot(rsi_res['Wealth'], label='RSI Pullback', color='#9467bd', lw=2)

plt.axhline(1.0, color='gray', linestyle=':', alpha=0.6)
plt.title('Multi-Strategy Equity Curves (Net of 10 bps Fees)', fontsize=15, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Portfolio Wealth ($1 initial)')
plt.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()

## Part 9 · Capstone Challenge: Architect Your Own Custom Strategy!

### The Challenge
Now it's your turn to be the quantitative portfolio manager! Combine the paradigms you've learned into your own **custom hybrid strategy**.

#### Recommended Concept: "Trend-Filtered RSI Dip Buying"
*Hypothesis:* Buying RSI oversold dips is risky in a downtrend (catching a falling knife). But buying oversold dips **during a confirmed uptrend** is one of the highest-probability swing trading setups.

**Suggested Rules:**
1. **Trend Filter:** Price must be above its 50-day EMA (`Close > EMA_50`).
2. **Entry Trigger:** If in an uptrend, enter Long when `RSI < 50` (pullback within uptrend).
3. **Exit Trigger:** Exit to Cash when `Close < EMA_50` (trend broken) OR when `RSI > 70` (overbought target reached).
4. **Execution:** Ensure 1-bar lag via `.shift(1)` and test using `backtest_strategy()`.

*Or feel free to invent your own rules!*

### Your Implementation
Write your complete custom strategy below, calculate its net performance, and see where it ranks on the leaderboard!

In [ ]:
df_capstone = df_market.copy()

# 9.1: Calculate indicators
df_capstone['EMA_50'] = df_capstone['Close'].ewm(span=50, adjust=False).mean()

delta = df_capstone['Close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()
df_capstone['RSI'] = 100 - (100 / (1 + (avg_gain / avg_loss.replace(0, np.nan))))

# 9.2: Custom Hybrid Rule: Buy pullbacks in confirmed uptrends
is_uptrend = df_capstone['Close'] > df_capstone['EMA_50']
oversold_dip = df_capstone['RSI'] < 50
exit_rule = (df_capstone['Close'] < df_capstone['EMA_50']) | (df_capstone['RSI'] > 70)

raw_custom_sig = np.where(
    is_uptrend & oversold_dip, 1,
    np.where(exit_rule, 0, np.nan)
)
df_capstone['Signal_Custom'] = pd.Series(raw_custom_sig, index=df_capstone.index).ffill().fillna(0)

# 9.3: Position with 1-bar lag
df_capstone['Position_Custom'] = df_capstone['Signal_Custom'].shift(1).fillna(0)

# 9.4: Backtest and metrics
custom_res = backtest_strategy(df_capstone, 'Position_Custom', fee_bps=10)
custom_metrics = calculate_metrics(custom_res['Wealth'], custom_res['Net_Return'], "My Custom Hybrid Strategy")

print(pd.Series(custom_metrics))

## Part 10 · Summary & Practical Quant Takeaways

### What You Have Accomplished
In this lab, you successfully built a complete end-to-end quantitative trading pipeline:
1. **Eliminated Lookahead Bias:** You mastered why `df['Position'] = df['Signal'].shift(1)` is essential for reliable backtesting.
2. **Implemented 4 Foundational Paradigms:**
   - **Donchian Breakouts:** Trend-following via new highs/lows.
   - **Buffered Dual EMAs:** Trend-following with noise/whipsaw filters.
   - **Z-Score Mean Reversion:** Statistical mean-reverting entry with asymmetric mean exits.
   - **RSI from Scratch:** Oscillating momentum dynamics.
3. **Engineered a Reusable Backtesting Engine:** Factored in execution friction (10 bps fees) and tracked portfolio wealth.
4. **Calculated Risk-Adjusted Metrics:** Measured CAGR, Sharpe ratio, and Maximum Drawdown.
5. **Architected a Custom Strategy:** Translated a compound trading hypothesis into working Python code.

### Next Steps for Live Strategy Development
- **Lecture 04 & 05 Preview:** How to incorporate regression, multi-factor risk models, and time-series forecasting.
- **Lecture 07 Preview:** Rigorous walk-forward optimization, out-of-sample cross-validation, and avoiding parameter over-fitting.